In [2]:
import os
import json
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
import stanza

# ===== PATH SETUP =====
BASE_DIR = "Scientific_Novelty_Detection_2022_2025"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
TRIPLET_DIR = os.path.join(BASE_DIR, "Triplets", "SKG")
TEMP_DIR = os.path.join(BASE_DIR, "temp_pdf")

os.makedirs(TRIPLET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

TASKS = ["Dia", "MT", "NLI", "Par", "QA", "SA", "Sum"]

GROBID_URL = "http://localhost:8070/api/processFulltextDocument"

# ===== STANZA =====
nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma", use_gpu=False, tokenize_batch_size=32)

2026-02-26 00:03:10 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-02-26 00:03:11 INFO: Downloaded file to C:\Users\spars\stanza_resources\resources.json
2026-02-26 00:03:11 WARNING: Language en package default expects mwt, which has been added
2026-02-26 00:03:11 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-02-26 00:03:11 INFO: Using device: cpu
2026-02-26 00:03:11 INFO: Loading: tokenize
2026-02-26 00:03:15 INFO: Loading: mwt
2026-02-26 00:03:15 INFO: Loading: pos
2026-02-26 00:03:18 INFO: Loading: lemma
2026-02-26 00:03:19 INFO: Done loading processors!


In [3]:
def load_skg_metadata(task):
    path = os.path.join(CACHE_DIR, f"SKG_{task}_metadata.json")
    with open(path, "r") as f:
        return json.load(f)

In [4]:
def download_pdf(url, save_path):
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(r.content)
            return True
    except Exception:
        pass
    return False

In [5]:
def parse_with_grobid(pdf_path):
    with open(pdf_path, "rb") as f:
        r = requests.post(
            GROBID_URL,
            files={"input": f}
        )
    if r.status_code == 200:
        return r.text
    return None

In [6]:
def extract_text_from_tei(tei_xml):
    try:
        root = ET.fromstring(tei_xml)
    except:
        return ""

    texts = []
    for elem in root.iter():
        if elem.text:
            cleaned = elem.text.strip()
            if cleaned:
                texts.append(cleaned)

    return " ".join(texts)

In [7]:
def extract_sentences_chunked(text, chunk_size=3000):

    sentences = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]

        try:
            doc = nlp(chunk)
            for sent in doc.sentences:
                sentence_text = " ".join([token.text for token in sent.tokens])
                sentences.append(sentence_text)
        except:
            continue

    return sentences

In [8]:
def extract_triplets_from_sentences(sentences, paper_id, task, year):

    rows = []

    for idx, sentence in enumerate(sentences):

        words = sentence.split()

        if len(words) >= 3:
            sub = words[0]
            pred = words[1]
            obj = " ".join(words[2:5])

            rows.append({
                "topic": task,
                "paper_ID": paper_id,
                "sentence_ID": idx,
                "info-unit": "auto",
                "sub": sub,
                "pred": pred,
                "obj": obj,
                "triplets": f"{sub} {pred} {obj}",
                "pred_weights": None,
                "year": year
            })

    return rows

In [9]:
def build_skg_triplets(task):

    metadata = load_skg_metadata(task)
    output_path = os.path.join(TRIPLET_DIR, f"{task}_triplets_results.csv")
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"SKG_{task}_triplet_checkpoint.json")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint = json.load(f)
        start_idx = checkpoint["index"]
        print(f"Resuming {task} from index {start_idx}")
    else:
        start_idx = 0

    for i in tqdm(range(start_idx, len(metadata))):

        paper = metadata[i]
        paper_id = paper["id"].split("/")[-1]
        year = paper.get("year")
        pdf_url = paper.get("pdf_url")

        if not pdf_url:
            continue

        temp_pdf = os.path.join(TEMP_DIR, f"temp_{task}_{i}.pdf")

        if not download_pdf(pdf_url, temp_pdf):
            continue

        tei_xml = parse_with_grobid(temp_pdf)
        os.remove(temp_pdf)

        if not tei_xml:
            continue

        text = extract_text_from_tei(tei_xml)
        sentences = extract_sentences_chunked(text)

        triplet_rows = extract_triplets_from_sentences(
            sentences, paper_id, task, year
        )

        if triplet_rows:
            df = pd.DataFrame(triplet_rows)
            df.to_csv(
                output_path,
                mode="a",
                header=not os.path.exists(output_path),
                index=False
            )

        # Save checkpoint (index only)
        with open(checkpoint_path, "w") as f:
            json.dump({"index": i + 1}, f)

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"{task} SKG triplets completed.")

In [10]:
for task in TASKS:
    build_skg_triplets(task)

Resuming Dia from index 112


100%|██████████| 38/38 [14:26<00:00, 22.81s/it]


Dia SKG triplets completed.


100%|██████████| 150/150 [1:29:09<00:00, 35.66s/it]


MT SKG triplets completed.


100%|██████████| 150/150 [1:38:14<00:00, 39.30s/it] 


NLI SKG triplets completed.


  3%|▎         | 5/150 [00:43<20:58,  8.68s/it]


KeyboardInterrupt: 

In [11]:
build_skg_triplets("PAR")

100%|██████████| 150/150 [1:55:51<00:00, 46.34s/it]  

PAR SKG triplets completed.


In [12]:
build_skg_triplets("QA")

100%|██████████| 150/150 [2:00:01<00:00, 48.01s/it]  

QA SKG triplets completed.


In [13]:
build_skg_triplets("SA")

100%|██████████| 150/150 [1:44:15<00:00, 41.70s/it]  

SA SKG triplets completed.


In [14]:
build_skg_triplets("SUM")

100%|██████████| 150/150 [1:30:53<00:00, 36.36s/it]  

SUM SKG triplets completed.
